# Real Estate Appraisal System

An end-to-end machine learning workflow for predicting residential property prices.

**Pipeline:** Data Cleaning → EDA → Correlation Analysis → Feature Engineering → Model Training → Evaluation → Model Selection

This notebook mirrors the logic in `src/train.py`, presented step by step with visualizations for exploration and reporting purposes.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

pd.set_option("display.max_columns", None)

## 1. Load Data

In [ ]:
df = pd.read_csv("../data/housing_data.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Data Cleaning

We check for missing values, duplicate rows, and outliers in the target variable (`price`).

In [ ]:
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

print(f"\nDuplicate rows: {df.duplicated().sum()}")

In [ ]:
df_clean = df.drop_duplicates().copy()

# Impute missing numeric values with the median
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_clean[col].isna().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Remove extreme price outliers using the IQR method
q1, q3 = df_clean["price"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 3 * iqr, q3 + 3 * iqr
before = len(df_clean)
df_clean = df_clean[(df_clean["price"] >= lower) & (df_clean["price"] <= upper)].reset_index(drop=True)

print(f"Rows before cleaning: {len(df)}")
print(f"Rows after cleaning:  {len(df_clean)}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_clean["price"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of Price")

sns.boxplot(x=df_clean["price"], ax=axes[1], color="lightcoral")
axes[1].set_title("Price Boxplot")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.scatterplot(data=df_clean, x="square_footage", y="price", ax=axes[0, 0], alpha=0.5)
axes[0, 0].set_title("Price vs Square Footage")

sns.boxplot(data=df_clean, x="bedrooms", y="price", ax=axes[0, 1])
axes[0, 1].set_title("Price vs Bedrooms")

sns.boxplot(data=df_clean, x="neighborhood_quality", y="price",
            order=["Low", "Medium", "High", "Premium"], ax=axes[1, 0])
axes[1, 0].set_title("Price vs Neighborhood Quality")

sns.boxplot(data=df_clean, x="condition", y="price",
            order=["Poor", "Fair", "Good", "Excellent"], ax=axes[1, 1])
axes[1, 1].set_title("Price vs Condition")

plt.tight_layout()
plt.show()

## 4. Correlation Matrix

In [ ]:
numeric_df = df_clean.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Matrix - Numeric Features")
plt.tight_layout()
plt.show()

In [ ]:
corr["price"].sort_values(ascending=False)

## 5. Feature Engineering

We create a few derived features that often help models capture non-linear relationships:
- `rooms_total`: bedrooms + bathrooms
- `lot_to_house_ratio`: lot size relative to living area
- `amenity_score`: count of pool / garden / renovation / garage
- `is_new`: whether the house is 5 years old or newer

In [ ]:
df_feat = df_clean.copy()
df_feat["rooms_total"] = df_feat["bedrooms"] + df_feat["bathrooms"]
df_feat["lot_to_house_ratio"] = df_feat["lot_size"] / df_feat["square_footage"]
df_feat["amenity_score"] = (
    df_feat["has_pool"] + df_feat["has_garden"] + df_feat["renovated"] + (df_feat["garage_spaces"] > 0).astype(int)
)
df_feat["is_new"] = (df_feat["house_age"] <= 5).astype(int)

df_feat[["rooms_total", "lot_to_house_ratio", "amenity_score", "is_new"]].head()

## 6. Train / Test Split

In [ ]:
NUMERIC_FEATURES = [
    "square_footage", "bedrooms", "bathrooms", "lot_size", "house_age",
    "distance_to_city_center", "garage_spaces", "has_pool", "has_garden",
    "school_rating", "crime_rate", "renovated", "rooms_total",
    "lot_to_house_ratio", "amenity_score", "is_new",
]
CATEGORICAL_FEATURES = ["neighborhood_quality", "condition", "property_type"]
TARGET = "price"

X = df_feat[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df_feat[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 7. Model Training

We train and compare three regression models:
1. **Linear Regression** — simple, interpretable baseline
2. **Random Forest Regressor** — bagged ensemble of decision trees
3. **Gradient Boosting Regressor** — boosted ensemble, typically strongest on tabular data

All models share the same preprocessing pipeline (scaling + one-hot encoding).

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42),
}

fitted_pipelines = {}
results = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}
    fitted_pipelines[name] = pipe
    print(f"{name:20s} | MAE: {mae:>10,.0f} | RMSE: {rmse:>10,.0f} | R2: {r2:.4f}")

## 8. Performance Comparison

In [ ]:
results_df = pd.DataFrame(results).T.sort_values("R2", ascending=False)
results_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
results_df["MAE"].plot(kind="bar", ax=axes[0], color="steelblue", title="MAE (lower is better)")
results_df["RMSE"].plot(kind="bar", ax=axes[1], color="indianred", title="RMSE (lower is better)")
results_df["R2"].plot(kind="bar", ax=axes[2], color="seagreen", title="R2 Score (higher is better)")
for ax in axes:
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
best_name = results_df.index[0]
best_pipeline = fitted_pipelines[best_name]
print(f"Best model: {best_name}")
print(results_df.loc[best_name])

## 9. Feature Importance (Best Tree-Based Model)

If the best model is tree-based (Random Forest or Gradient Boosting), we can inspect which features it relied on most.

In [ ]:
if hasattr(best_pipeline.named_steps["model"], "feature_importances_"):
    ohe_cols = best_pipeline.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES)
    all_feature_names = NUMERIC_FEATURES + list(ohe_cols)
    importances = best_pipeline.named_steps["model"].feature_importances_

    feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False).head(15)

    plt.figure(figsize=(10, 7))
    feat_imp.sort_values().plot(kind="barh", color="darkorange")
    plt.title(f"Top 15 Feature Importances - {best_name}")
    plt.tight_layout()
    plt.show()
else:
    print("Best model does not expose feature_importances_ (e.g. Linear Regression).")

## 10. Save the Best Model

The full pipeline (preprocessing + trained model) is serialized with `pickle`/`joblib` so it can be
loaded directly by the Streamlit app (`app.py`) or reused elsewhere without retraining.

In [ ]:
import joblib
import json
from pathlib import Path

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(best_pipeline, MODELS_DIR / "best_model.pkl")

metadata = {
    "best_model": best_name,
    "results": results,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "categorical_options": {col: sorted(df_feat[col].unique().tolist()) for col in CATEGORICAL_FEATURES},
}
with open(MODELS_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved model and metadata to ../models/")

## 11. Sanity Check: Predict on a Sample Property

In [ ]:
sample = X_test.iloc[[0]]
predicted = best_pipeline.predict(sample)[0]
actual = y_test.iloc[0]

print("Sample input:")
print(sample.T)
print(f"\nPredicted price: ${predicted:,.0f}")
print(f"Actual price:    ${actual:,.0f}")
print(f"Difference:      ${abs(predicted - actual):,.0f}")

## 12. Conclusion

- The **Gradient Boosting Regressor** typically achieves the strongest R² and lowest error on this dataset, followed by Random Forest, with Linear Regression as a solid interpretable baseline.
- Key price drivers include square footage, neighborhood quality, condition, and school rating.
- The saved model (`models/best_model.pkl`) powers the interactive Streamlit prediction app (`app.py`).

See the project README for setup instructions and further details.